# Adding a MeVPrtl/HNL sample (Optional)

Skip this notebook entirely unless your analysis needs a MeVPrtl/BSM (e.g.
HNL) generator sample on top of your BNB MC sample.

| Section | File | What you set |
|---|---|---|
| B1 | -- | produce `MCHNL_FILE` via `run_df_maker.py` |
| B2 | `analysis.py` | `signal_categories`, `background_categories`, `define_signal_hnl` |
| B3 | `preprocess.py` | copy in `preprocess_mchnl` |
| B4 | `io.py` | copy in `load_mchnl` |
| B5 | `plotting.py` | copy in `plot_mc_hnl` |
| B6 | `analysis.py` | cuts for `load_mchnl` |
| B7 | `preprocess.py` | new functions (fix/derive variables) |

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
# TODO: set to the PARENT directory of your cafpybara/ clone -- NOT cafpybara/ itself (skipped if already importable).
sys.path.append("TODO_SET_ME/parent_dir_of_cafpybara")

# TODO: set to your cafpyana clone (skipped if already exported).
os.environ.setdefault("CAFPYANA_PATH", "TODO_SET_ME/path/to/cafpyana")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cafpybara import core

# TODO: import your own analysis's name instead of _template.
import cafpybara.analyses._template as ana

STILL_ON_TEMPLATE = ana.__name__.endswith('_template')
if STILL_ON_TEMPLATE:
    print("Still importing _template directly -- see the TODO above.")
else:
    print("Imported cleanly -- a real, importable analysis.")

## B1. Make `MCHNL_FILE` with `cafpyana`

Prepare your MeVPrtl dataframe with `cafpyana`'s `run_df_maker.py`.

This example below uses a `cafpyana` config to process MeVPrtl (e.g. HNL) files without any selections applied:
1. `cd $CAFPYANA_PATH`
2. ```bash
   python run_df_maker.py -c ./analysis_village/hnl_nuee_nupi0/configs/hnl_mcmevprtl_nopreselect_savepfp.py -l <input.list> -o <output_name>
   ```

Already made files from this example can be found here: `/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/demo`

For more information, see: https://github.com/sungbinoh/cafpyana/blob/lnguyen/hnl_analysis/analysis_village/hnl_nuee_nupi0/README.md

**TODO:** set `MC_FILE` (your BNB MC sample) and `MCHNL_FILE` (produced above).

In [ ]:
#MC_FILE = "/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/demo/mcbnb_noselection.df"
MC_FILE = "/path/to/your_mc_file.h5"

#MCHNL_FILE = "/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/demo/mchnl_nupi0_m240_noselection.df"
MCHNL_FILE = "/path/to/your_mchnl_file.h5"

## B2. Set HNL signal categories in `analysis.py`

* **TODO:**
  * add an `"hnl"` entry to the shared `signal_categories` dict in `/my_analysis/analysis.py`
  * add a `background_categories` view (same dict, minus `"hnl"`) for B5's `plot_mc_hnl`
  * add `'background_categories'` to `__all__`

> ⚠️ Adding `"hnl"` here renumbers `signal`/`cosmic` (0/1 -> 1/2). `"hnl"` now carries 0 as the signal.

Example:
```python
__all__ = ['signal_categories', 'signal_dict', 'background_categories', 'DEFAULT_CUTS', 'define_signal']

signal_categories = {
    "hnl": {"value": 0, "label": "HNL", "color": "#4B0082"},
    "signal":  {"value": 1, "label": "Neutrino", "color": "#E7004C"},
    "cosmic":  {"value": 2, "label": "Cosmic",   "color": "#708090"},
}
signal_dict = {k: v["value"] for k, v in signal_categories.items()}

#anything that is not your signal
background_categories = {k: v for k, v in signal_categories.items() if k != "hnl"}
```

* **TODO:**
  * set `define_signal_hnl` in `/my_analysis/analysis.py`
  * add `'define_signal_hnl'` to `__all__`

Example:
```python
__all__ = ['signal_categories', 'signal_dict', 'background_categories', 'DEFAULT_CUTS', 'define_signal', 'define_signal_hnl']

def define_signal_hnl(indf):
    indf["signal"] = signal_dict["hnl"]
    return indf
```

## B3. Copy in `preprocess_mchnl`

* **TODO:** copy into `/my_analysis/preprocess.py`

Example:
```python
def preprocess_mchnl(df: pd.DataFrame) -> pd.DataFrame:
    df = _core_preprocess_mc(df)
    return df
```

## B4. Copy in `load_mchnl`

`load_mchnl` doesn't ship in `_template` by default -- copy it in.

* **TODO:**
  * copy `load_mchnl` into `/my_analysis/io.py`
  * import `define_signal_hnl`/`preprocess_mchnl`
  * add `'load_mchnl'` to `__all__`

Example:
```python
from ...core.io import load_mevprtl as _core_load_mevprtl
from .analysis import define_signal_hnl
from .preprocess import preprocess_mchnl

__all__ = ['load_mc', 'load_data', 'load_mchnl']

def load_mchnl(
    file: str,
    keys: list | None = None,
    mevprtl_key: str = 'mevprtl',
    cuts=None,
    rec_key: str = REC_KEY,
    preprocess_fn=preprocess_mchnl,
    define_signal_fn=define_signal_hnl,
):
    return _core_load_mevprtl(
        file, rec_key=rec_key, preprocess_fn=preprocess_fn, define_signal_fn=define_signal_fn,
        keys=keys, mevprtl_key=mevprtl_key, cuts=cuts,
    )
```

See below for the actual call:

In [ ]:
# What you'd actually write in your own analysis notebook:
try:
    mc_df, mc_pot, mc_ngen = ana.load_mc(MC_FILE)
    hnl_df, hnl_pot, hnl_info = ana.load_mchnl(MCHNL_FILE)
    print("HNL |U|^2:", hnl_info["simU"], " mass [MeV]:", hnl_info["hnlM"])
except Exception as e:
    mc_df, hnl_df = None, None
    print("No real MC file here (this notebook needs none) -- expected:")
    print(f"  {type(e).__name__}: {e}")

## B5. Copy in `plot_mc_hnl`

`plot_mc_hnl` doesn't ship in `_template` by default -- copy it in.

* **TODO:**
  * copy `plot_mc_hnl` into `/my_analysis/plotting.py`
  * add `'plot_mc_hnl'` to `__all__`

> ⚠️ `categories` defaults to the full `signal_categories` (includes `hnl`), which double-legends `HNL` alongside `hnl_categories` -- pass `categories=ana.background_categories` (set in B2) for the MC stack instead.

Example:
```python
__all__ = ['plot_var', 'plot_mc_data', 'plot_mc_hnl']

def plot_mc_hnl(mc_df, hnl_df, var, bins, **kwargs):
    kwargs = _inject_defaults(kwargs)
    return _core_plotting.plot_mc_hnl(mc_df, hnl_df, var, bins, **kwargs)
```

Example: hnlpi0's per-mass legend relabeling via `hnl_categories_for_mass` (`analyses/hnlpi0/analysis.py`)

See below for the actual call:

In [ ]:
#VAR = "your_variable"
#BINS = [0, 1, 2, 3]
VAR = ('pfp', 'shw', 'bestplane_energy', '', '', '') #TEST
BINS = np.linspace(0, 1, 10) #TEST

if hnl_df is not None:
    # mc_df/mc_pot/hnl_df/hnl_pot loaded at step B4 above
    try:
        ana.plot_mc_hnl(
            mc_df, hnl_df, VAR, bins=BINS,
            scale_hnl=mc_pot / hnl_pot,
            categories=ana.background_categories,
            hnl_categories={"hnl": ana.signal_categories["hnl"]},
        )
    except Exception as e:
        print("plot_mc_hnl not copied in yet (see the TODO above) -- expected:")
        print(f"  {type(e).__name__}: {e}")
else:
    print("No real MC file here (this notebook needs none) -- see B4 above")

## B6. Apply cuts to `load_mchnl`

* **TODO:** Apply `DEFAULT_CUTS` (set in B2's `analysis.py`, same list as your BNB MC):
  * Method 1: `load_mchnl(cuts=...)` -- cuts each chunk while loading; use for the final selected sample.
  * Method 2: `select(hnl_df, cuts, stage=..., savedict=...)` -- on an already-loaded `hnl_df`; use to stop at a given cut (`stage=`) or get every stage for a cut flow (`savedict=True`).

In [ ]:
# Method 1: Apply cuts when loading mc_df/hnl_df, overlay as in B5
try:
    mc_df_cut, mc_pot_cut, mc_ngen_cut = ana.load_mc(MC_FILE, cuts=ana.DEFAULT_CUTS)
    hnl_df_cut, hnl_pot_cut, hnl_info_cut = ana.load_mchnl(MCHNL_FILE, cuts=ana.DEFAULT_CUTS)
    ana.plot_mc_hnl(
        mc_df_cut, hnl_df_cut, VAR, bins=BINS,
        scale_hnl=mc_pot_cut / hnl_pot_cut,
        categories=ana.background_categories,
        hnl_categories={"hnl": ana.signal_categories["hnl"]},
    )
except Exception as e:
    print("No real MC file here (this notebook needs none) -- expected:")
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# Method 2: select() on an already-loaded hnl_df (stage=/savedict= control how far to apply cuts)
if hnl_df is not None:
    #hnl_df is the original, uncut dataframe loaded at step B4
    hnl_df_dict = ana.select(hnl_df, ana.DEFAULT_CUTS, savedict=True)
    for name, df_stage in hnl_df_dict.items():
        print(f"  after {name!r}: {len(df_stage)} rows")
else:
    print("No real MC file here (this notebook needs none) -- see B4 above")

### Judging a cut's effect with `plot_var`

`plot_var` has display kwargs useful for comparing before/after a cut --
see `core.plotting.plot_var`'s docstring for the full list.

* `percents=True` -- append each category's %-of-total to its legend
  label; compare pre-/post-cut to see which category a cut removes most.
* `normalize=True` -- rescale each histogram so its integral is 1;
  compare *shape* when a cut also changes the overall rate a lot.
* `counts=True` -- append raw event counts to the legend instead.

See below for the actual call:

In [ ]:
if hnl_df is not None:
    #mc_df/hnl_df are the original, uncut dataframes loaded at step B4
    mc_df_dict = ana.select(mc_df, ana.DEFAULT_CUTS, savedict=True)
    hnl_df_dict = ana.select(hnl_df, ana.DEFAULT_CUTS, savedict=True)
    mc_df_after = list(mc_df_dict.values())[-1]
    hnl_df_after = list(hnl_df_dict.values())[-1]

    ana.plot_mc_hnl(
        mc_df, hnl_df, VAR, bins=BINS,
        scale_hnl=mc_pot / hnl_pot,
        categories=ana.background_categories,
        hnl_categories={"hnl": ana.signal_categories["hnl"]},
        percents=True, title="Before cuts",
    )
    plt.show()

    ana.plot_mc_hnl(
        mc_df_after, hnl_df_after, VAR, bins=BINS,
        scale_hnl=mc_pot / hnl_pot,
        categories=ana.background_categories,
        hnl_categories={"hnl": ana.signal_categories["hnl"]},
        percents=True, title="After cuts",
    )
    plt.show()
else:
    print("No real MC file here (this notebook needs none) -- see B4 above")

## B7. Add new functions in `preprocess.py` (Optional)

* **TODO (optional):** Add new functions to `preprocess_mchnl` in
  `/my_analysis/preprocess.py` to fix existing variables, compute new variables, etc.
  This automatically applies when calling `load_mchnl`.

Example: Add MeVPrtl timing calibration (`analyses/hnlpi0/preprocess.py`)

```python
from ...core.preprocess import fix_timing_calibration
from ...core import timing_calibration as tc

def preprocess_mchnl(df):
    df = _core_preprocess_mc(df)
    df = fix_timing_calibration(df, period=tc.mchnl_period_calib,
                                 t0_offset=tc.mchnl_offset_calib)
    return df
```

Description of function `fix_timing_calibration`:
* Input: `slc.barycenterFM.flashTime` + `slc.vertex.z`, plus your
  sample's `period`/`t0_offset` calibration constants
  (`core.timing_calibration`).
* Output: adds `slc.barycenterFM.flashTime_calib` (+ period-folded
  `_calib_mod`) columns.

In [ ]:
if hnl_df is not None:
    VAR = ('slc', 'barycenterFM', 'flashTime_calib_mod', '', '', '') #New variable to plot from the fix_timing_calibration() function
    BINS = np.linspace(0, 19, 20) #TEST

    #mc_df/hnl_df are the original, uncut dataframes loaded at step B4
    mc_df_dict = ana.select(mc_df, ana.DEFAULT_CUTS, savedict=True)
    hnl_df_dict = ana.select(hnl_df, ana.DEFAULT_CUTS, savedict=True)
    mc_df_after = list(mc_df_dict.values())[-1]
    hnl_df_after = list(hnl_df_dict.values())[-1]

    ana.plot_mc_hnl(
        mc_df_after, hnl_df_after, VAR, bins=BINS,
        scale_hnl=mc_pot / hnl_pot,
        categories=ana.background_categories,
        hnl_categories={"hnl": ana.signal_categories["hnl"]},
        percents=True, title="After cuts",
    )
    plt.show()
else:
    print("No real MC file here (this notebook needs none) -- see B4 above")